# Module 5 — Machine learning for biosignals

**Who this is for:** you can process a signal and now need to classify it — or you have done ML before, but not on biosignals.

> **Self-check — skip this module if you can already:**
>
> - [ ] build a feature matrix `X` with one row per epoch, plus `y` and `groups`
> - [ ] put a scaler and a classifier in a `Pipeline`
> - [ ] explain why `GroupKFold` and not a random split, and show the difference in numbers
> - [ ] report a confusion matrix, macro-F1 and Cohen's kappa, not just accuracy
>
> If you can tick every box, go to the next module. If not, work through the parts you can't tick. Before
> jumping to a section, run the module's **setup cell** (the first code cell): later sections use variables
> defined earlier, so run the cells above a section if you get a `NameError`.

**You will learn**
- features from epochs, and what to do with unusable ones
- scikit-learn's fit / predict pattern and Pipelines
- leakage-safe validation by subject
- metrics for imbalanced classes
- class imbalance and seeds
- where deep learning fits

**Where the course uses this:** Chapters 11–14, Labs 11–14 and the design clinic, and `evaluate()` in every capstone track.

*Run each cell with **Shift+Enter**, top to bottom. Nothing here needs data files; once the libraries are
installed (see the [overview](README.md#setup-first)), the notebook runs offline.*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/python_tutorial/05_machine_learning.ipynb)
&nbsp; Part of the **Python for biosignal processing** tutorial — see the [overview](README.md) to choose which modules you need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import trapezoid      # area under a curve (band power)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay)
SEED = 2013
rng = np.random.default_rng(SEED)

## 1. A small synthetic cohort

12 subjects, 60 epochs each, three classes defined by the frequency of a rhythm. Each subject has their own
**signal offset** (like electrode impedance) and their own **shift in rhythm frequency** (people really differ,
e.g. in their individual alpha frequency). That is what makes biosignal ML hard: a model can learn the
*person* instead of the *class*.

In [ ]:
fs, epoch_s = 100, 4
n_subj, n_ep = 12, 60
t = np.arange(epoch_s * fs) / fs
X_raw, y, groups = [], [], []
for s in range(n_subj):
    offset = rng.normal(0, 1.5)                       # subject-specific, unrelated to the class
    shift = rng.normal(0, 2.0)                        # subject-specific frequency shift (Hz)
    for k in range(n_ep):
        c = rng.choice(3, p=[0.6, 0.3, 0.1])           # imbalanced classes
        f0 = [6, 10, 14][c] + shift + rng.normal(0, 0.7)
        e = np.sin(2 * np.pi * f0 * t) + (1.0 + 0.3 * abs(offset)) * rng.standard_normal(t.size) + offset
        X_raw.append(e); y.append(c); groups.append(s)
X_raw, y_raw, groups_raw = np.array(X_raw), np.array(y), np.array(groups)   # the originals: never overwritten
print(X_raw.shape, np.bincount(y_raw))

## 2. From epochs to features

Turn each epoch into a short row of numbers you can explain: band powers from a Welch PSD, plus simple
statistics. `X` has shape `(n_epochs, n_features)`.

Real recordings contain **unusable epochs**: flat lines when a sensor drops out, gaps (NaN), very short
pieces. A feature function must not quietly turn those into numbers (a flat epoch has zero power, and
`log(0)` is `-inf`). Here the policy is: **check each epoch, reject the ones that fail, and keep a record of
which were rejected and why**. Whatever policy you choose, write it down and make it testable.

Two details matter on real data:

- **Units.** The same EEG can be stored in volts (MNE) or microvolts, a factor of a million. A check like
  "total power below 10⁻¹²" would call a perfectly good signal in volts "flat". The flatline test below
  compares the signal with *itself*, so it gives the same answer in any unit.
- **A quality check is not a numerical fix.** Deciding an epoch is unusable is one thing; keeping `log()` away
  from zero is another. They get separate, named settings.

In [ ]:
BANDS = [(4, 8), (8, 12), (12, 16)]          # Hz; both edges lie on the 1 Hz Welch grid below

def is_flat(e, rel_tol=1e-6):
    """True if the epoch hardly varies compared with its own size (a sensor dropout, not just a small
    signal). Unit-independent: multiplying e by any constant gives the same answer."""
    scale = np.abs(e).max()
    return scale == 0 or np.ptp(e) <= rel_tol * scale

def epoch_features(e, fs, log_floor_rel=1e-12):
    """Features of one epoch: log band powers, relative band powers, mean and std.

    e: 1-D epoch in any amplitude unit (V, µV, a.u.); band powers come out in that unit squared.
    log_floor_rel: numerical floor for the logarithm, as a fraction of the epoch's total power.
    Raises ValueError for an epoch that is too short, non-finite or flat."""
    e = np.asarray(e, dtype=float)
    if e.ndim != 1 or e.size < fs:
        raise ValueError(f"need a 1-D epoch of at least {fs} samples, got shape {e.shape}")
    if not np.isfinite(e).all():
        raise ValueError("epoch contains NaN or inf")
    if is_flat(e):
        raise ValueError("flat epoch (the signal hardly varies)")
    f, P = signal.welch(e, fs=fs, nperseg=fs)
    total = trapezoid(P, f)
    bp = np.array([trapezoid(P[(f >= lo) & (f <= hi)], f[(f >= lo) & (f <= hi)]) for lo, hi in BANDS])
    return np.r_[np.log(bp + log_floor_rel * total), bp / total, e.mean(), e.std()]

names = ["logP_4-8", "logP_8-12", "logP_12-16", "rel_4-8", "rel_8-12", "rel_12-16", "mean", "std"]

# What happens to bad epochs:
for bad, why in [(np.zeros(4 * fs), "flat"), (np.full(4 * fs, 5.0), "flat at an offset"),
                 (np.r_[np.ones(399), np.nan], "NaN"), (np.ones(50), "short")]:
    try:
        epoch_features(bad, fs)
    except ValueError as err:
        print(f"{why:18s} rejected: {err}")

# Same physical signal in microvolts and in volts: both accepted, and the relative features are identical.
s10 = np.sin(2 * np.pi * 10 * np.arange(4 * fs) / fs)
f_uv, f_v = epoch_features(1.0 * s10, fs), epoch_features(1e-6 * s10, fs)
assert np.allclose(f_uv[3:6], f_v[3:6])
print("10 Hz sinusoid accepted in µV and in V; relative band powers identical")

Two honest caveats about these checks:

- `is_flat` is a **simple, scale-invariant heuristic**, not a sensor-dropout detector. Because it compares the
  variation with the signal's largest absolute value, a huge DC offset can make a real signal look "flat".
  Real quality control depends on the sensor's units, resolution, typical offsets and the task.
- Accepting a signal in either unit is not the same as giving the **same features**. Only the *relative* band
  powers match; the log powers shift (by about −27.6 from µV to V) and the mean and std scale by a million.
  Bring every recording to the **same physical unit before combining them**. `StandardScaler` rescales numbers
  statistically; it does not fix inconsistent units.

Now apply the policy to the whole cohort. The collection is a function of the **original** arrays
(`X_raw`, `y_raw`, `groups_raw`), so re-running it always gives the same result, and the rejection log keeps
the original epoch number, subject and reason.

In [ ]:
def collect_features(X_raw, y_raw, groups_raw, fs):
    rows, kept, rejected = [], [], []
    for i, e in enumerate(X_raw):
        try:
            rows.append(epoch_features(e, fs))
            kept.append(i)
        except ValueError as err:
            rejected.append({"epoch": i, "subject": int(groups_raw[i]), "reason": str(err)})
    if not rows:
        raise ValueError(f"No usable epochs remain; inspect the rejection log: {rejected[:3]} ...")
    kept = np.array(kept, dtype=int)
    X = np.array(rows)
    assert np.isfinite(X).all()
    return X, y_raw[kept], groups_raw[kept], rejected

X, y, groups, rejected = collect_features(X_raw, y_raw, groups_raw, fs)
print(X.shape, f"({len(rejected)} epochs rejected)")

Test the policy on the three cases that matter: nothing rejected (above), some rejected (and the cell run
twice), and everything rejected:

In [ ]:
tiny_X = np.stack([np.zeros(4 * fs), X_raw[0], X_raw[1]])        # first epoch flat
tiny_y, tiny_g = np.array([1, 2, 0]), np.array([0, 0, 1])
for attempt in (1, 2):                                             # running it twice gives the same answer
    Xt, yt, gt, rej = collect_features(tiny_X, tiny_y, tiny_g, fs)
    assert yt.tolist() == [2, 0] and gt.tolist() == [0, 1] and rej[0]["epoch"] == 0
print("some rejected: labels and subjects stay aligned, and a re-run is safe; log:", rej)

try:
    collect_features(np.zeros((2, 4 * fs)), np.array([0, 1]), np.array([0, 0]), fs)
except ValueError as err:
    print("all rejected:", str(err)[:60], "...")

## 3. scikit-learn in one pattern: `fit`, then `predict`

Every model works the same way. Put preprocessing (like scaling) **inside a Pipeline**, so it is re-fitted
on the training part of every split and never sees the test data. Here we train on subjects 0–8 and predict
for subjects 9–11, whom the model has never seen:

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
train, test = groups < 9, groups >= 9
model.fit(X[train], y[train])
print("predicted:", model.predict(X[test][:8]))
print("true:     ", y[test][:8])

## 4. Honest validation: split by subject

The question in the course is almost always *"how well does this work on a **new** person?"* So the test
folds must contain subjects the model never saw. That is `GroupKFold` with `groups`. A random split
(`KFold` with shuffling) mixes epochs of the same person into train and test and **flatters** the model.

Below, the splits are made explicitly and handed to `cross_val_predict`, so it is obvious which splitter
uses the subject IDs. Each epoch is predicted once, by a model that never saw it: these are **out-of-fold
(OOF) predictions**.

In [ ]:
random_splits = list(KFold(5, shuffle=True, random_state=SEED).split(X))
subject_splits = list(GroupKFold(5).split(X, y, groups))
for tr, te in subject_splits:
    assert set(groups[tr]).isdisjoint(groups[te])          # no subject in both train and test

def evaluate(splits, model):
    """Pooled out-of-fold predictions, and kappa / macro-F1 computed on them."""
    pred = cross_val_predict(model, X, y, cv=splits)
    return pred, cohen_kappa_score(y, pred), f1_score(y, pred, average="macro")

rf = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=200, random_state=SEED,
                                                           class_weight="balanced"))
_, k_rand, f_rand = evaluate(random_splits, rf)
pred, k_grp, f_grp = evaluate(subject_splits, rf)
print(f"random split  : pooled OOF kappa {k_rand:.3f}  macro-F1 {f_rand:.3f}   <- optimistic")
print(f"split by subj : pooled OOF kappa {k_grp:.3f}  macro-F1 {f_grp:.3f}   <- the honest number")

The gap between those two lines is **leakage**: with a random split the model can recognise the *person*
(here, their signal offset and frequency shift) instead of the class. How big the gap is depends on the
data; it can be small, or even reversed. The reason to split by subject is the question you are answering
(a new person), not the size of the gap. In the capstone, only the grouped number counts, and each track's
`evaluate()` enforces it for you.

## 5. Metrics for imbalanced classes

With about 60 % of epochs in one class, a model that always says "class 0" gets about 60 % accuracy and
learns nothing. Report the **confusion matrix** first, then metrics that account for imbalance.

In [ ]:
always0 = np.zeros_like(y)
print(f"'always class 0': accuracy {accuracy_score(y, always0):.2f}, balanced {balanced_accuracy_score(y, always0):.2f}, kappa {cohen_kappa_score(y, always0):.2f}")
print(f"random forest  : accuracy {accuracy_score(y, pred):.2f}, balanced {balanced_accuracy_score(y, pred):.2f}, kappa {k_grp:.2f}")

ConfusionMatrixDisplay(confusion_matrix(y, pred)).plot(colorbar=False)
plt.title("random forest, split by subject (pooled OOF)"); plt.show()

| Metric | Use it for |
|---|---|
| Confusion matrix | Always first: shows *which* classes are confused |
| Cohen's kappa | Agreement beyond chance; standard for sleep staging |
| Macro-F1 | Average F1 over classes, each class counts equally |
| Balanced accuracy | Mean recall per class |
| Plain accuracy | Only with balanced classes, and never alone |

## 6. Per-fold scores and their spread

A pooled score hides how much performance varies between subjects. Score each fold separately as well, and
look at which subjects each fold tested:

In [ ]:
fold_kappas = []
for i, (tr, te) in enumerate(subject_splits):
    rf.fit(X[tr], y[tr])
    k = cohen_kappa_score(y[te], rf.predict(X[te]))
    fold_kappas.append(k)
    print(f"fold {i}: test subjects {sorted(set(groups[te].tolist()))}, "
          f"class counts {np.bincount(y[te], minlength=3).tolist()}, kappa {k:.3f}")
print(f"\nmean of fold kappas = {np.mean(fold_kappas):.3f}, SD across folds = {np.std(fold_kappas):.3f}")
print(f"pooled OOF kappa    = {k_grp:.3f}")

Two different numbers, both legitimate:

- **Pooled OOF kappa** — computed once over all out-of-fold predictions.
- **Mean of fold kappas ± SD** — the average of per-fold scores. Folds here have different subjects and class
  mixes, and kappa is not an average-friendly quantity, so this need not equal the pooled value.

Say which one you report. The SD across folds **describes how much performance varies** between groups of
subjects. It is not a confidence interval: folds share training data, so they are not independent repeats.

## 7. Practical notes

- **Seeds:** set `random_state=` on every model and splitter that has one, and seed NumPy. With the same seed
  and the same library versions, results are repeatable on one machine; across different hardware they can
  still differ in the last digits.
- **Class imbalance:** start with `class_weight="balanced"`. Resampling such as SMOTE (`imbalanced-learn`)
  must happen inside the training folds only, using `imblearn.pipeline.Pipeline` (scikit-learn's own
  `Pipeline` does not accept resamplers).
- **Feature selection and tuning** are also "learning from data": do them inside the folds (a Pipeline, or
  nested cross-validation), never on the whole dataset first. Keep any final hold-out set out of all of this.
- **Start simple:** a logistic regression or random forest on good features is a strong baseline. Beat it
  before reaching for anything bigger.
- **Deep learning:** Chapter 14 and Lab 14 compare a small neural network (`MLPClassifier`) with a forest.
  Frameworks such as PyTorch and Keras are beyond this course; on small cohorts a well-featured forest
  often wins.

## Try it yourself

Replace the random forest with `LogisticRegression(max_iter=2000, class_weight="balanced")` inside the
same Pipeline. Compare pooled kappa and macro-F1 under the subject split. Then fit it on all the data and
list, for each class, the three features with the **largest absolute** weights, with their sign.
(Hint: `model[-1].coef_`, `model[-1].classes_` and `names`.)

When you read the weights, keep it modest: they are coefficients for *standardised* features, they describe
associations *given all the other features*, and correlated features (the log and relative power of the same
band) can share or swap weight. The largest weight is not the strongest physiological cause, nor a reliable
ranking of feature importance.

In [ ]:
# TODO

**Solution:** try it yourself first, then compare with the [solutions notebook](solutions.ipynb) (Module 5). Yours may differ and still be right.

**Next:** Module 6 — habits that keep a project reproducible and debuggable.